In [5]:
# st_count = pd.read_csv("/mnt/d/ST_Graduation_Project_data/STARmap/combined_spatial_count.txt", sep="\t")
# print(st_count.shape)
# st_location = pd.read_csv("/mnt/d/ST_Graduation_Project_data/STARmap/combined_Locations.txt", sep="\t")
# st_density = pd.read_csv("/mnt/d/ST_Graduation_Project_data/STARmap/combined_spot_clusters.txt", sep="\t", index_col=0)
# adata = sc.AnnData(X=st_count)
# adata.uns['density'] = st_density
# adata.obsm['spatial'] = st_location.values
# print(adata)
# adata.write_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad")

In [6]:
# import scanpy as sc
# import pandas as pd
# adata =sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad")
# adata.obs['cell_type']=adata.obs['celltype']
# print(adata.obs['cell_type'].value_counts())
# adata.write_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad")
# adata_st = sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad")
# st_label = adata_st.uns['density']
# st_label_df = pd.DataFrame(st_label)
# st_label_df.to_csv("/mnt/d/ST_Graduation_Project/SC_MAP_ST/evaluate/STARmap_density.csv")

In [ ]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad" \
    --n_epochs 150 \
    --output_dir ../deconv_results/STARmap
#    --marker_selection_method l1 
#    --precomputed_marker_file "../deconv_results/STARmap/final_genes.txt"

Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4
   Batch size: 1024
   Epochs: 150
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 128
   Loss type: MSE
   Lambda MMD: 0.05
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad


In [6]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad" \
    --stage1_model_path "../deconv_results/STARmap/final_vae.pth" \
    --output_dir "../deconv_results/STARmap/" \
    --n_epoch 250

Sample name: Spatial
Stage 1 model: ../deconv_results/STARmap/final_vae.pth
Output directory: ../deconv_results/STARmap/
Device: cpu
Weight threshold: 0.001
Loading pretrained VAE Encoder...
   VAE architecture: 775 -> 128
   Output type: mse
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 14249 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/STARmap/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([86, 128])
   Cluster expressions (marker): torch.Size([86, 775])
   Cluster expressions (all genes): 86 × 882
   Loaded celltype mapping: 86 clusters
   Average cell counts: 1594647.9
Loaded all genes list: 882 genes
VAE Encoder loaded: 775 -> 128
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '5

GAT Training: 100%|██████████| 250/250 [01:44<00:00,  2.38epoch/s, Total=5.3691, MSE=0.8519, Spot_Cosine=0.2737, Gene_Cosine=0.2136, Pearson=0.6204, Gene_Pearson=0.5476, P_pat=5, M_pat=25, C_pat=3]  


Evaluating model results...
Applying weight threshold: 0.001
   Non-zero elements: 5670 -> 4550 (28.0%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   ✅ Scale factor computed from MARKER genes (775 genes)
      Spot marker total (real): mean=2810.7
      Reconstructed marker total: mean=873.0
      Scale factor: min=0.2450, max=6.6703, mean=3.1779
   Saved: ../deconv_results/STARmap//Spatial_reconstructed_all_genes.csv
   Cell type composition...
   Using cluster→celltype mapping from Stage 1 checkpoint (86 entries).
   Found duplicate celltype names: ['Astro', 'Inhibitory Other', 'Excitatory L2/3', 'Inhibitory Sst', 'Olig', 'Other', 'Excitatory L5', 'Excitatory L6', 'Inhibitory Vip', 'Inhibitory Pvalb', 'Excitatory L4']. Merging corresponding cluster columns by summing weights.
   Columns before: 86, after merge: 15
   Saved cell composition (celltype): ../deconv_results/STARmap//Spatial_cell_composition.csv
